# 01 · Build the training dataset

Streams RAID and DACTYL, draws a balanced ~400k-row sample stratified by
generator, domain and label, extracts the 30 features, and writes two Parquet
files.

**Two datasets, because there are two models (PRD 8.1).**

| | rows | adversarial |
|---|---|---|
| `train_a.parquet` | Model A, the strict detector | included, ~15% of the mix |
| `train_b.parquet` | Model B, the surrogate | excluded entirely |

Excluding the adversarial rows from B is not an oversight — it is what makes B
a faithful stand-in for third-party detectors, which have no such hardening.

**The labelling rule that matters (E8).** RAID's paraphrase / synonym /
homoglyph variants are the humanized-text augmentation, free. Humanized *AI*
text stays labelled AI. Humanized *human* text stays labelled **human** —
labelling a rewritten human essay as AI would teach the model that editing is
evidence of machine authorship, which is exactly how detectors end up
punishing careful writers and non-native speakers (R2, A4).

Runtime: 40–90 minutes, mostly feature extraction. CPU is fine.


In [ ]:
# Kaggle setup. Run once per session.
!pip install -q "transformers>=4.44" "datasets>=2.20" sentencepiece onnx onnxruntime \
    "optimum[onnxruntime]" pyarrow

import sys, os
from pathlib import Path

GIT_URL = "https://github.com/ByteCraft-9/ai-text-humanizer.git"   # <-- the missing line

# The repo is added as a Kaggle dataset, or cloned. Point REPO at it.
REPO = Path("/kaggle/input/ai-detector-repo") if Path("/kaggle/input/ai-detector-repo").exists() \
       else Path("/kaggle/working/ai-detector")
if not REPO.exists():
    !git clone --depth 1 $GIT_URL /kaggle/working/ai-detector

sys.path.insert(0, str(REPO / "training"))
sys.path.insert(0, str(REPO / "api"))

WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)
DATA = WORK / "data"; DATA.mkdir(exist_ok=True)
MODELS = WORK / "models"; MODELS.mkdir(exist_ok=True)
print("repo:", REPO)

In [ ]:
from pathlib import Path
from lib.data import SampleSpec, build_dataset, feature_statistics

# Start small to verify the pipeline end to end, then raise to 400_000.
SPEC = SampleSpec(total=400_000, adversarial_share=0.15, human_share=0.5)

frame_a = build_dataset(DATA / "train_a.parquet", SPEC, include_adversarial=True)


In [ ]:
spec_b = SampleSpec(total=SPEC.total, adversarial_share=0.0, human_share=0.5, seed=SPEC.seed + 1)
frame_b = build_dataset(DATA / "train_b.parquet", spec_b, include_adversarial=False)


In [ ]:
# Gate: class balance and domain coverage must be verified before training
# (PRD 18, phase 1). A skewed sample produces a model that looks fine on its
# own validation split and fails on everything else.
import pandas as pd

for name, frame in (("A", frame_a), ("B", frame_b)):
    print(f"--- Model {name}: {len(frame):,} rows")
    print(frame["label"].value_counts(normalize=True).round(3).to_dict())
    print("  generators:", frame["generator"].nunique(), " domains:", frame["domain"].nunique())
    print("  adversarial share:", round(frame["adversarial"].mean(), 3))
    print("  words: median", int(frame["text"].str.split().str.len().median()))

balance = frame_a["label"].mean()
assert 0.4 < balance < 0.6, f"Model A is unbalanced: {balance:.3f} positive"
assert frame_b["adversarial"].sum() == 0, "Model B must contain no adversarial rows"
print("\nGate passed.")


In [ ]:
# Feature standardisation statistics. Paste these into
# api/_lib/features.py (FEATURE_MEAN / FEATURE_STD) so inference standardises
# with the same numbers training used. The placeholders shipped there are
# rough estimates, and leaving them in place costs real accuracy.
import json
stats = feature_statistics(frame_a)
print(json.dumps(stats, indent=2)[:800])
(MODELS / "feature_stats.json").write_text(json.dumps(stats, indent=2))
